# 🚀 The Ultimate Kaggle Starter Template (XGBoost Edition)
**Welcome to your competition journey!** This notebook is designed to take you from raw data to a submittable model using the industry-standard XGBoost algorithm.

> **Tip for Beginners:** Cells marked with `[EDIT REQUIRED]` are places where you'll need to change code to match your specific dataset. Everything else should work out of the box!

---

### 📑 Table of Contents
0. [Kaggle Auth & Download (Colab Only)](#auth)
1. [Environment & Library Imports](#setup)
2. [Data Loading & Inspection](#loading)
3. [Deep Exploratory Data Analysis (EDA)](#eda)
4. [Advanced Graphical Presentation](#viz)
5. [Preprocessing & Feature Engineering](#prep)
6. [Model Training with XGBoost](#modeling)
7. [Performance Evaluation](#eval)
8. [Submission Production](#submit)

<a id='auth'></a>
## 🔑 Step 0: Kaggle Authentication (Colab Only)

If you are running this on **Google Colab**, you need to give it access to your Kaggle account to download the data. 

**Instructions:**
1. Go to your Kaggle account settings and click **"Create New API Token"**. This downloads a `kaggle.json` file.
2. Run the cells below to upload that file.

In [ ]:
# --- OPTION 1: Upload kaggle.json manually ---
try:
    from google.colab import files
    print("Please upload your kaggle.json file:")
    files.upload()
except ImportError:
    print("Not running in Colab. Skipping manual upload.")

In [1]:
# --- OPTION 2: Use kaggle.json from Google Drive ---
# (Uncomment the lines below if you prefer to load from Drive)

from google.colab import drive
drive.mount('/content/drive')
!cp drive/MyDrive/kaggle.json .
!cp drive/MyDrive/kaggle.zip .
!if [ -f kaggle.json ]; then exit 1; fi; unzip kaggle.zip

MessageError: Failed to issue request POST https://colab.research.google.com/tun/m/credentials-propagation/m-s-32xzb5yuqn3le?authtype=dfs_ephemeral&version=2&dryrun=false&propagate=true&record=false&authuser=0&authuser=0: Bad Request
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 400 (Bad Request)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpage/error_logo-15

In [ ]:
# --- Setting up the Kaggle Directory ---
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("✅ Kaggle credentials configured!")

In [ ]:

# --- [EDIT REQUIRED] Download Dataset ---
COMPETITION_NAME = 'titanic' # [CHANGE REQUIRED] Change 'titanic' to your competition name!

!kaggle competitions download -c {COMPETITION_NAME}
!unzip -o {COMPETITION_NAME}.zip # -o overwrites existing files

<a id='setup'></a>
## 🛠 1. Environment & Library Imports

To build a model, we need several tools:
*   `pandas` & `numpy` for handling data tables.
*   `matplotlib` & `seaborn` for drawing graphs.
*   `xgboost` for the powerful Extreme Gradient Boosting model.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

# Machine Learning libraries
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Settings to make it look pretty
%matplotlib inline
plt.style.use('fivethirtyeight')
sns.set_palette("viridis")
warnings.filterwarnings('ignore') # Hides annoying warnings
pd.set_option('display.max_columns', None) # Show all columns in tables

print("🚀 Global settings initialized. Ready for analysis.")

<a id='loading'></a>
## 📁 2. Data Loading & Inspection

In Kaggle, you usually get two files:
1.  **Train (`train.csv`)**: Data used to teach the model (includes the answers).
2.  **Test (`test.csv`)**: Data where you need to predict the answers.

In [ ]:
# Try to load the data
try:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
    print(f"✅ Success! Found {len(train_df)} training rows and {len(test_df)} testing rows.")
except FileNotFoundError:
    print("❌ Files not found! Did you run Step 0 or upload the CSVs?")
    train_df = test_df = None

if train_df is not None:
    display(train_df.head()) # Show the first 5 row
    print(train_df.columns)

<a id='eda'></a>
## 📊 3. Exploratory Data Analysis (EDA)

Before building a model, we must "meet" the data. We check for **Missing Values** (NaNs) and see what the numbers look like.

In [ ]:
if train_df is not None:
    print("--- Data Types & Non-Null Counts ---")
    print(train_df.info())
    
    print("\n--- Statistical Overview ---")
    display(train_df.describe().T)
    
    print("\n--- Missing Values Count ---")
    nulls = train_df.isnull().sum()
    print(nulls[nulls > 0])

<a id='viz'></a>
## 🎨 4. Advanced Graphical Presentation

**What are we looking at?**
We've optimized these plots to be **MAXIMUM SIZE** for readability, even with 30+ features.

### 📈 4.1 Target Distribution
Checking how our "answers" are spread out. Continuous targets are split into 36 parts.

In [ ]:
if train_df is not None:
    # [EDIT REQUIRED] Update target_name to your Target column!
    target_name = 'Survived'
    
    unique_vals = train_df[target_name].nunique()
    # 🤖 Auto-detect task type: Regression if numeric & many unique values, else classification
    is_classification = not (pd.api.types.is_numeric_dtype(train_df[target_name]) and unique_vals > 35)
    
    plt.figure(figsize=(24, 10)) # Ultra-wide for distribution
    
    if not is_classification:
        sns.histplot(train_df[target_name], bins=36, kde=True, color='teal')
        plt.title(f'Target Analysis: {target_name} (Continuous, 36 Bins)', fontsize=24)
    else:
        sns.countplot(data=train_df, x=target_name, palette='viridis')
        plt.title(f'Target Analysis: {target_name} (Categorical)', fontsize=24)
        
    plt.xlabel(target_name, fontsize=18)
    plt.ylabel('Count', fontsize=18)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.show()

### 🗺️ 4.2 Feature Correlation Matrix
This massive square map shows how every feature talks to every other feature. Great for spotting redundant data!

In [ ]:
if train_df is not None:
    numeric_cols = train_df.select_dtypes(include=[np.number])
    
    # Huge Square Figure for clarity
    plt.figure(figsize=(25, 25))
    
    # Calculate correlation
    corr = numeric_cols.corr()
    
    # Plotting
    sns.heatmap(corr, 
                annot=True, 
                cmap='RdBu_r', 
                fmt='.2f', 
                square=True, 
                linewidths=1.0, 
                cbar_kws={"shrink": .8},
                annot_kws={"size": 12, "weight": "bold"}) # Bigger, bolder fonts
    
    plt.title('The Feature Relationship Map', fontsize=30, pad=30)
    plt.xticks(fontsize=16, rotation=90)
    plt.yticks(fontsize=16)
    
    plt.show()

<a id='prep'></a>
## 🛠 5. Preprocessing & Feature Engineering

Machine Learning models don't like text (like "Male"/"Female") or missing values. 
We use a combination of **Custom Logic** and a **Pipeline** to automatically handle data:

### Part 1: Smart Automated Feature Engineering
1.  **Missing Value Indicators**: If any column has missing values, we create a `{col}_isMissing` feature (1 if missing, 0 if present) to help the model learn from the *fact* that data was missing.
2.  **Low Correlation Pruning**: For numeric columns, if the correlation with the target is extremely weak (absolute value < 0.05), we drop them. Less noise helps the model focus on true signals!
3.  **High Cardinality Dropping**: If a categorical (text) column has too many unique values (e.g., > 50% of the dataset size), it's likely an ID or a name and will overcomplicate the model. We drop it. Otherwise, we keep it to process normally.

### Part 2: The Sklearn Pipeline
1.  Fill in remaining missing numbers with the **Median**.
2.  Turn text categories into **Numbers** (One-Hot Encoding).
3.  Scale numbers so they are all in the same range.


In [ ]:
def clean_and_feature(df, is_train=False):
    """Local function for custom automated data cleaning and feature engineering."""
    df = df.copy()
    
    # 1. Missing value indicators
    # Creates a helper column for every column that has empty values
    for col in df.columns:
        if df[col].isnull().any():
            df[f"{col}_isMissing"] = df[col].isnull().astype(int)
    
    # 2. Drop standard ID and high-text columns
    drops = [c for c in ['PassengerId', 'Name', 'Id', 'id', 'Ticket'] if c in df.columns]
    if len(drops) > 0:
        print(f"🔹 Dropping basic ID/text columns: {drops}")
        df = df.drop(columns=drops)

    # 3. Handle mixed types in categorical columns (e.g., bool + strings)
    # This prevents TypeError in scikit-learn's OneHotEncoder during fitting
    cat_check_cols = df.select_dtypes(include=['object', 'bool']).columns
    for col in cat_check_cols:
        if col != target_name:
            # Convert values to strings while preserving actual NaNs so Imputer can find them
            df[col] = df[col].map(lambda x: str(x) if pd.notnull(x) else x)

    # 4. Drop low correlation numeric columns (Train set only logic)
    global train_df, target_name
    if target_name in df.columns:
        # We are processing the training set
        corr_drops = []
        for col in df.select_dtypes(include=['number']).columns:
            if col != target_name:
                corr = df[col].corr(df[target_name])
                if abs(corr) < 0.02: # Set threshold to 0.02
                    corr_drops.append(col)
        
        if len(corr_drops) > 0:
            print(f"📉 Dropping low correlation (< 0.02) numeric columns: {corr_drops}")
            df = df.drop(columns=corr_drops)
            # Save for the test set, so we perform the EXACT same drops later
            global _saved_corr_drops
            _saved_corr_drops = corr_drops
    else:
        # We are processing the test set
        try:
            if len(_saved_corr_drops) > 0:
                drops_here = [c for c in _saved_corr_drops if c in df.columns]
                df = df.drop(columns=drops_here)
        except NameError:
            pass
            
    # 5. Drop high-cardinality metadata (names, specific cabin numbers, etc.)
    obj_drops = []
    total_rows = len(df)
    for col in df.select_dtypes(include=['object']).columns:
        if col == target_name: continue
        num_unique = df[col].nunique()
        if num_unique > 0.6 * total_rows:
            obj_drops.append(col)
            
    if len(obj_drops) > 0:
        print(f"🗑️ Dropping high-cardinality object columns (> 60% unique): {obj_drops}")
        df = df.drop(columns=obj_drops)
        
    return df

if train_df is not None:
    # Process the data
    train_clean = clean_and_feature(train_df)
    
    # IMPROVED: Define features by type, strictly EXCLUDING the target column
    # This prevents the model from accidentally "seeing" the answer
    num_features = train_clean.select_dtypes(include=['number']).columns.drop(target_name, errors='ignore')
    cat_features = train_clean.select_dtypes(include=['object']).columns.drop(target_name, errors='ignore')

    # --- Automatic Processing Engine ---
    # Numeric: Fill gaps with Medium, then scale result
    num_proc = Pipeline([('imputer', SimpleImputer(strategy='median')), 
                         ('scaler', StandardScaler())])
                         
    # Categorical: Fill missing with 'NA', then convert to binary numbers
    cat_proc = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
                         ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

    # Assemble the final transformation blueprint
    transformer = ColumnTransformer(transformers=[
        ('num', num_proc, num_features),
        ('cat', cat_proc, cat_features)
    ])
    
    print("✅ Preprocessing engine is ready and robust!")


<a id='modeling'></a>
## 🤖 6. Model Training with XGBoost

XGBoost is the winning algorithm for most Kaggle tabular competitions. It uses "Gradient Boosting" to improve from its mistakes step-by-step.

In [ ]:
if train_df is not None:
    # Separate the data (X) from the answer (y)
    X = train_clean.drop(columns=[target_name])
    y = train_clean[target_name]

    # --- XGBoost Parameters Explained ---
    xgb_params = {
        'n_estimators': 500,        # How many trees to build. More = more complex model.
        'learning_rate': 0.05,     # Step size. Smaller values make training slower but more precise.
        'max_depth': 6,            # How deep each tree can go. Higher = more prone to overfitting.
        'subsample': 0.8,          # Fraction of rows used for each tree (prevents overfitting).
        'colsample_bytree': 0.8,   # Fraction of columns used for each tree.
        'random_state': 42,
        
        # --- GPU Acceleration ---
        # Uncomment the line below to use Kaggle/Colab GPU (requires GPU runtime enabled)
        # 'tree_method': 'gpu_hist'
    }

    # Create a Master Pipeline (Process Data -> Predict Answer)
    ModelClass = xgb.XGBClassifier if is_classification else xgb.XGBRegressor
    print(ModelClass)
    final_model = Pipeline([
        ('prep', transformer),
        ('model', ModelClass(**xgb_params))
    ])

    # Split data to test the model before submitting
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    # Learn from the training data
    final_model.fit(X_train, y_train)
    
    print(f"✅ XGBoost training finished!")

<a id='eval'></a>
## 🔍 7. Performance Evaluation

How well did we do? We check the accuracy score and the **Confusion Matrix** to see where the model got confused.

Now work For both Classifier and Regressor: 

In [ ]:
if train_df is not None:
    y_pred = final_model.predict(X_val)
    
    if is_classification:
        acc = accuracy_score(y_val, y_pred)
        print(f"🎯 Validation Accuracy: {acc:.2%}")
        
        # Visualization: heatmap of success vs failure
        plt.figure(figsize=(12, 10)) # Larger evaluation square
        sns.heatmap(confusion_matrix(y_val, y_pred), annot=True, fmt='d', cmap='Greens', square=True, annot_kws={"size": 16})
        plt.xlabel('Prediction', fontsize=18)
        plt.ylabel('Actual Truth', fontsize=18)
        plt.title('Confusion Matrix (Task Type: Classification)', fontsize=22)
        plt.show()
    else:
        # Calculate Metrics
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)

        print(f"🏠 RMSE (Avg Error): ${rmse:,.2f}")
        print(f"📉 MAE: ${mae:,.2f}")
        print(f"📈 R2 Score: {r2:.4f}")

        # Visualization: Actual vs Predicted
        plt.figure(figsize=(12, 8))
        plt.scatter(y_val, y_pred, alpha=0.5, color='teal')
        plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'k--', lw=2) # Diagonal line
        plt.xlabel(f'Actual {target_name}', fontsize=18)
        plt.ylabel(f'Predicted {target_name}', fontsize=18)
        plt.title(f'Actual vs Predicted {target_name} (Task Type: Regression)', fontsize=22)
        plt.show()


<a id='submit'></a>
## 📤 8. Submission Production

Final Step! Apply the same logic to the `test.csv` file and save it to a file that Kaggle accepts.

In [ ]:
if test_df is not None:
    # Clean the test data the same way as train data
    test_clean = clean_and_feature(test_df)
    
    # Predict!
    predictions = final_model.predict(test_clean)
    
    # --- UNIVERSAL FIX ---
    # Smart conversion: If the original target was Boolean, convert predictions to Boolean
    if train_df[target_name].dtype == 'bool' or is_classification:
        # Check if the competition specifically wants True/False (like Spaceship Titanic)
        # You can manually set this to True, or let the code attempt to match the train type
        final_predictions = predictions.astype(bool) if train_df[target_name].dtype == 'bool' else predictions
    else:
        final_predictions = predictions

    # Create the submission file
    submission_file = pd.DataFrame({
        "Id": test_df.iloc[:, 0], 
        target_name: final_predictions
    })
    # ---------------------

    submission_file.to_csv('submission.csv', index=False)
    print("🎉 DONE! 'submission.csv' is ready.")


In [ ]:
# Submit the generated CSV file to the Kaggle competition
!kaggle competitions submit -c spaceship-titanic -f submission.csv -m "XGBoost model with engineered Title, Deck, and FamilySize features"
####################EDIT#HERE##^^^^^^^^^^^^^^^^^##EDIT#HERE######

100% 2.77k/2.77k [00:00<00:00, 8.41kB/s]
Successfully submitted to Titanic - Machine Learning from Disaster